# Multi-provider smoke notebook — cantus v0.2.0

**Audience:** release manager (run by hand before tagging v0.2.0).

Exercises the two Tier 2 adapters shipped in v0.2.0 (OpenAI, Anthropic) end-to-end against the real provider endpoints. Cassettes are NOT used here — this notebook is the human gate.

**Setup before running:**

```bash
pip install 'cantus[providers]'
pip install python-dotenv  # secret loader; not a cantus dependency
# Create .env next to this notebook:
#   OPENAI_API_KEY=sk-...
#   ANTHROPIC_API_KEY=sk-ant-...
```

Tier 2 `stream()` yields text deltas only (no tool-call delta in v0.2.0) — that's verified at the bottom.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import cantus
print('cantus version:', cantus.__version__)
assert cantus.__version__.startswith('0.2.'), 'run against v0.2.x'

## OpenAI — chat (one round trip)

In [ ]:
from cantus import load_chat_model, Message

openai_chat = load_chat_model('openai/gpt-4o-mini')
resp = openai_chat.chat(
    [Message(role='user', content='Reply with exactly the word: cantus')],
)
print('content :', resp.message.content)
print('stop    :', resp.stop_reason)
print('usage   :', resp.usage)
assert resp.message.content.strip(), 'OpenAI returned empty content'

## Anthropic — chat (one round trip)

In [ ]:
anthropic_chat = load_chat_model('anthropic/claude-sonnet-4-6')
resp = anthropic_chat.chat(
    [Message(role='user', content='Reply with exactly the word: cantus')],
    max_tokens=64,
)
print('content :', resp.message.content)
print('stop    :', resp.stop_reason)
print('usage   :', resp.usage)
assert resp.message.content.strip(), 'Anthropic returned empty content'

## OpenAI — stream (text deltas only)

In [ ]:
deltas = []
for chunk in openai_chat.stream([Message(role='user', content='Count from 1 to 5.')]):
    assert isinstance(chunk, str), 'stream must yield str only — got ' + type(chunk).__name__
    deltas.append(chunk)
    print(chunk, end='', flush=True)
print('\n---')
print(f'{len(deltas)} deltas received')

## Anthropic — stream (text deltas only)

In [ ]:
deltas = []
for chunk in anthropic_chat.stream(
    [Message(role='user', content='Count from 1 to 5.')],
    max_tokens=128,
):
    assert isinstance(chunk, str), 'stream must yield str only — got ' + type(chunk).__name__
    deltas.append(chunk)
    print(chunk, end='', flush=True)
print('\n---')
print(f'{len(deltas)} deltas received')

## End-to-end — bridge into Agent

Tier 2 → Tier 1 round trip via `ChatModelAsHandle`. Agent code is unchanged from v0.1.x — it only sees the Tier 1 protocol.

In [ ]:
from cantus import Agent, ChatModelAsHandle, skill

@skill
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

agent = Agent(model=ChatModelAsHandle(anthropic_chat, system='You are terse.'))
result = agent.run('What is 17 plus 25?')
print('final answer:', result.final_answer)